In [2]:
%%sql
-- FIXED: Use actual silver_budget columns
CREATE OR REPLACE TEMPORARY VIEW temp_silver AS 
SELECT date, site, department, 
       monthly_budget_eur, monthly_actual_eur, variance_pct, budget_status,
       revenue_eur, expense_eur, spend_efficiency
FROM silver_budget;

SELECT COUNT(*) as row_count FROM temp_silver;


StatementMeta(, cee14193-42c9-4bca-97bd-23c52e7b27ec, 4, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
%%pyspark

df = spark.sql("""
    SELECT date, site, department, 
           monthly_budget_eur, monthly_actual_eur, 
           variance_pct, budget_status, 
           revenue_eur, expense_eur, spend_efficiency 
    FROM silver_budget
""").toPandas()

print(f"✅ Loaded {len(df)} rows")
print("Columns:", df.columns.tolist())
print(df[['site', 'department', 'variance_pct']].head())



StatementMeta(, cee14193-42c9-4bca-97bd-23c52e7b27ec, 9, Finished, Available, Finished)

✅ Loaded 600 rows
Columns: ['date', 'site', 'department', 'monthly_budget_eur', 'monthly_actual_eur', 'variance_pct', 'budget_status', 'revenue_eur', 'expense_eur', 'spend_efficiency']
        site  department  variance_pct
0     Dublin   Marketing        -58.71
1   Limerick  Operations        -58.57
2     Galway       SYGMA        -18.46
3  Waterford  Operations        -16.70
4   Limerick       SYGMA        -16.61


In [7]:
%%pyspark
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import numpy as np

# ML features (your actual columns)
features = ['variance_pct', 'monthly_actual_eur', 'monthly_budget_eur', 'spend_efficiency']
X = df[features].fillna(df[features].mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Production anomaly detection
iso_forest = IsolationForest(contamination=0.1, random_state=42)
df['iso_anomaly'] = iso_forest.fit_predict(X_scaled)
df['iso_score'] = iso_forest.decision_function(X_scaled)

print("🏆 Isolation Forest Results:")
print(df['iso_anomaly'].value_counts())
print("\n🚨 TOP ANOMALIES:")
anomalies = df[df['iso_anomaly'] == -1][['site', 'department', 'variance_pct', 'iso_score']]
print(anomalies.sort_values('iso_score').head())


StatementMeta(, cee14193-42c9-4bca-97bd-23c52e7b27ec, 10, Finished, Available, Finished)

🏆 Isolation Forest Results:
iso_anomaly
 1    540
-1     60
Name: count, dtype: int64

🚨 TOP ANOMALIES:
          site      department  variance_pct  iso_score
524  Waterford  US_Foodservice         86.54  -0.233554
449     Dublin  US_Foodservice         79.47  -0.194955
349     Dublin           SYGMA         97.56  -0.186379
174  Waterford   International         86.34  -0.181112
124  Waterford  US_Foodservice         78.50  -0.174997


In [8]:
%%pyspark
from scipy import stats

df['z_variance'] = np.abs(stats.zscore(df['variance_pct'].fillna(0)))
df['zscore_anomaly'] = (df['z_variance'] > 2.5).astype(int)

df['business_anomaly'] = (np.abs(df['variance_pct']) > 25).astype(int)

print(f"Z-Score anomalies: {df['zscore_anomaly'].sum()}")
print(f"Business rule (>25%): {df['business_anomaly'].sum()}")


StatementMeta(, cee14193-42c9-4bca-97bd-23c52e7b27ec, 11, Finished, Available, Finished)

Z-Score anomalies: 27
Business rule (>25%): 37


In [13]:
%%pyspark
df = df.sort_values(['site', 'department', 'date'])
df['predicted_next'] = df.groupby(['site', 'department'])['monthly_actual_eur'].transform(
    lambda x: x.rolling(3, min_periods=1).mean().shift(-1)
)
df['pred_error_pct'] = np.abs((df['predicted_next'] - df['monthly_actual_eur']) / 
                             df['monthly_actual_eur'] * 100).fillna(0)


StatementMeta(, cee14193-42c9-4bca-97bd-23c52e7b27ec, 16, Finished, Available, Finished)

In [14]:
%%pyspark
def cfo_action(row):
    actions = []
    priority = 'LOW'
    
    # ML anomaly
    if row['iso_anomaly'] == -1:
        actions.append("🚨 ML_ANOMALY")
        priority = 'HIGH'
    
    # Extreme variance
    if row['business_anomaly'] == 1:
        if row['variance_pct'] > 25:
            actions.append(f"CUT_{row['variance_pct']:.0f}%")
        else:
            actions.append(f"BOOST_{abs(row['variance_pct']):.0f}%")
        priority = 'HIGH'
    
    # Prediction alert
    if row['pred_error_pct'] > 20:
        actions.append("TREND_BREAK")
        priority = 'MEDIUM'
    
    return '; '.join(actions) if actions else "OK", priority

df[['cfo_alert', 'priority']] = df.apply(cfo_action, axis=1, result_type='expand')

print("🚨 TOP 10 CFO ACTIONS:")
print(df[df['priority'] != 'LOW'][['site', 'department', 'variance_pct', 'cfo_alert', 'priority']].head(10))


StatementMeta(, cee14193-42c9-4bca-97bd-23c52e7b27ec, 17, Finished, Available, Finished)

🚨 TOP 10 CFO ACTIONS:
     site     department  variance_pct                             cfo_alert  \
427  Cork  International        -18.26                           TREND_BREAK   
105  Cork  International        -11.37                           TREND_BREAK   
577  Cork      Marketing        -48.15  🚨 ML_ANOMALY; BOOST_48%; TREND_BREAK   
567  Cork      Marketing          4.18                           TREND_BREAK   
521  Cork      Marketing         17.45                           TREND_BREAK   
181  Cork      Marketing        -12.58                           TREND_BREAK   
173  Cork      Marketing         78.63    🚨 ML_ANOMALY; CUT_79%; TREND_BREAK   
126  Cork      Marketing        -17.06                           TREND_BREAK   
25   Cork      Marketing        -58.64  🚨 ML_ANOMALY; BOOST_59%; TREND_BREAK   
101  Cork     Operations        -40.01  🚨 ML_ANOMALY; BOOST_40%; TREND_BREAK   

    priority  
427   MEDIUM  
105   MEDIUM  
577   MEDIUM  
567   MEDIUM  
521   MEDIUM  
181   M

In [15]:
%%pyspark
gold_df = df[['date', 'site', 'department', 'monthly_budget_eur', 'monthly_actual_eur', 
              'variance_pct', 'budget_status', 'iso_anomaly', 'zscore_anomaly', 
              'business_anomaly', 'cfo_alert', 'priority', 'pred_error_pct']]

spark_gold = spark.createDataFrame(gold_df.fillna(0))
spark_gold.write.mode("overwrite").saveAsTable("gold_ai_production")
print("✅ PRODUCTION GOLD TABLE: gold_ai_production")


StatementMeta(, cee14193-42c9-4bca-97bd-23c52e7b27ec, 18, Finished, Available, Finished)

✅ PRODUCTION GOLD TABLE: gold_ai_production
